### Kryptologie

[Video1](https://youtu.be/58ejmffYwx4) - [Video2](https://youtu.be/rJqu2Mzz_uk)

Die **Kryptologie** ist die Wissenschaft von der sicheren Übermittlung (und Speicherung) von Informationen. Sie besteht aus zwei Teilen:

- **Kryptographie:** Wie erstellt man eine Chiffre, also ein Verschlüsselungsverfahren?
- **Kryptoanalyse:** Wie knackt man eine Chiffre?

##### Ausgangssituation und Rollen

**Alice** will **Bob** eine Nachricht schicken. Dazu wandelt sie den **Klartext** mit einem **Schlüssel** in einen **Geheimtext** um (**Verschlüsselung**, auch **Chiffrierung**) und verschickt den Geheimtext. Bob wandelt den Geheimtext mit dem Schlüssel wieder in den Klartext um (**Entschlüsselung**, auch **Dechiffrierung**).

Es gibt zwei typische Angreifer:

- **Eve** (von *eavesdropper*, Lauscher) will die Nachricht **mitlesen**, ohne sie zu verändern.
- **Mallory** (von *malicious*, bösartig) will die Nachricht **verändern** oder sich als jemand anderes ausgeben.

```
  Alice                                                       Bob
Klartext --verschlüsseln--> Geheimtext --entschlüsseln--> Klartext
            (Schlüssel)         |           (Schlüssel)
                                |
                     Eve liest mit, Mallory verändert
```

Bei den klassischen Verfahren benutzen Alice und Bob **denselben** Schlüssel. Solche Verfahren heißen **symmetrisch**.

##### Sicherheitsziele

| Sicherheitsziel | Bedeutung | Wer greift an? |
|:--|:--|:--|
| **Vertraulichkeit** | Niemand außer dem Empfänger kann die Nachricht lesen. | Eve liest mit. |
| **Integrität** | Die Nachricht ist unterwegs nicht verändert worden. | Mallory verändert die Nachricht. |
| **Authentizität** | Die Nachricht stammt wirklich von dem angegebenen Absender. | Mallory gibt sich als Alice aus. |
| **Verbindlichkeit** | Der Absender kann später nicht abstreiten, die Nachricht geschickt zu haben. | Alice selbst will ihre Nachricht abstreiten. |

Verschlüsselung sorgt vor allem für **Vertraulichkeit**. Für die anderen Ziele gibt es weitere Techniken, zum Beispiel digitale Signaturen.

### Transposition 

Bei einer **Transposition** werden die Buchstaben des Klartexts nicht verändert, sondern nur **anders angeordnet**.

Die **Skytale** ist ein Verschlüsselungsverfahren aus der Antike. Sender und Empfänger besaßen jeweils einen Holzstab mit **gleichem Umfang**. Der Sender wickelte ein schmales Band spiralförmig um den Stab und schrieb die Nachricht längs auf das Band. Nach dem Abwickeln war der Text unlesbar. Erst auf einem Stab mit demselben Umfang konnte die Nachricht wieder gelesen werden. Der **Umfang** ist also der **Schlüssel**.

<img src='./bild1.png' width='401'>

##### Die Skytale als Tabelle

Wir können die Skytale mit einer Tabelle nachspielen. Der Umfang bestimmt, wie viele **Zeilen** rund um den Stab Platz haben.

1. Wir schreiben den Klartext **zeilenweise** in eine Tabelle mit der vereinbarten Zeilenzahl. Falls am Ende Platz bleibt, füllen wir mit `X` auf.
2. Wir lesen die Tabelle **spaltenweise** von oben nach unten aus. Das ist der Geheimtext, so wie er auf dem abgewickelten Band steht.

Beispiel: Klartext `INFORMATIKMACHTSPASS`, Schlüssel 4 Zeilen

| | | | | |
|:-:|:-:|:-:|:-:|:-:|
| I | N | F | O | R |
| M | A | T | I | K |
| M | A | C | H | T |
| S | P | A | S | S |

Spaltenweise gelesen: `IMMS NAAP FTCA OIHS RKTS`, also `IMMSNAAPFTCAOIHSRKTS`.

Zum **Entschlüsseln** schreiben wir den Geheimtext spaltenweise in eine Tabelle mit 4 Zeilen und lesen zeilenweise.

In [1]:
import math

def skytale(text, zeilen, spalten=None):
    if spalten is None:
        spalten = math.ceil(len(text) / zeilen)    # ohne Angabe: so wenige Spalten wie möglich
    if zeilen * spalten < len(text):
        raise ValueError('Der Text passt nicht in die Tabelle, spalten ist zu klein.')
    text = text + 'X' * (zeilen * spalten - len(text))    # mit X auffüllen
    geheim = ''
    for s in range(spalten):
        geheim += text[s::spalten]    # Spalte s von oben nach unten lesen
    return geheim

def skytale_entschluesseln(geheim, zeilen):
    klartext = ''
    for z in range(zeilen):
        klartext += geheim[z::zeilen]    # Zeile z von links nach rechts lesen
    return klartext

geheim = skytale('INFORMATIKMACHTSPASS', 4)
print(geheim)                                 
print(skytale_entschluesseln(geheim, 4)) 


IMMSNAAPFTCAOIHSRKTS
INFORMATIKMACHTSPASS


Wenn wir uns entschließen, den Klartext mit längeren Zeilen auf den Stab zu schreiben, ändert sich der Geheimtext. Der Empfänger benötigt trotzdem nur die Anzahl der Zeilen, um den Geheimtext zu entschlüsseln.

In [3]:
geheim = skytale('INFORMATIKMACHTSPASS', 4, spalten=6)
print(geheim)                                 
print(skytale_entschluesseln(geheim, 4))     

IACSNTHSFITXOKSXRMPXMAAX
INFORMATIKMACHTSPASSXXXX


##### Wie sicher ist die Skytale?

Es gibt nur wenige sinnvolle Schlüssel, denn die Zeilenzahl muss kleiner als die Textlänge sein. Ein Angreifer probiert einfach alle aus. Das nennt man einen **Brute-Force-Angriff**. Der richtige Schlüssel ist der, bei dem ein lesbarer Text herauskommt:

In [10]:
geheim = 'IMMSNAAPFTCAOIHSRKTS'
for zeilen in range(2, 11):
    print(zeilen, skytale_entschluesseln(geheim, zeilen))

2 IMNAFCOHRTMSAPTAISKS
3 ISATOSTMNPCIRSMAFAHK
4 INFORMATIKMACHTSPASS
5 IACSMAARMPOKSFITNTHS
6 IAOTMPISMFHSTSNCRAAK
7 IPHMFSMTRSCKNATAOSAI
8 IFRMTKMCTSASNOAIAHPS
9 ITTMCSMASONIAHASPRFK
10 ICMAMOSINHASARPKFTTS


### Monoalphabetische Substitution

Bei einer **Substitution** werden die Buchstaben des Klartexts durch andere Buchstaben **ersetzt**. Bei einer **monoalphabetischen Substitution** wird jeder Klartextbuchstabe im ganzen Text **immer durch denselben** Geheimtextbuchstaben ersetzt.

##### Cäsar-Chiffre

Bei der **Cäsar-Chiffre** wird jeder Buchstabe um einen festen Betrag im Alphabet verschoben. Der Betrag ist der Schlüssel. Nach `Z` geht es wieder bei `A` weiter. Mit dem Schlüssel 3 wird aus `A` ein `D`, aus `B` ein `E` und aus `Y` ein `B`.

```
Klartext:   A B C D E F G H I J K L M N O P Q R S T U V W X Y Z
Geheimtext: D E F G H I J K L M N O P Q R S T U V W X Y Z A B C
```

Zum Rechnen nummerieren wir die Buchstaben: `A` = 0, `B` = 1, …, `Z` = 25. Dann gilt:

- Verschlüsseln: Geheimbuchstabe = (Klarbuchstabe + Schlüssel) mod 26
- Entschlüsseln: Klarbuchstabe = (Geheimbuchstabe − Schlüssel) mod 26

`mod 26` bedeutet: der Rest bei der Division durch 26. So landen wir nach `Z` wieder bei `A`.

In [11]:
ALPHABET = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

def caesar(text, k):
    ergebnis = ''
    for zeichen in text:
        if zeichen in ALPHABET:
            nummer = (ALPHABET.index(zeichen) + k) % 26
            ergebnis += ALPHABET[nummer]
        else:
            ergebnis += zeichen    # Leerzeichen usw. bleiben stehen
    return ergebnis

print(caesar('HALLO', 3))     # KDOOR
print(caesar('KDOOR', -3))    # HALLO   Entschlüsseln = rückwärts verschieben

KDOOR
HALLO


##### Brute-Force-Angriff auf die Cäsar-Chiffre

Für die Cäsar-Chiffre gibt es nur **25 sinnvolle Schlüssel** (eine Verschiebung um 0 oder 26 ändert nichts). Die Menge aller Schlüssel nennt man den **Schlüsselraum** (engl. *key space*). Bei nur 25 Schlüsseln probieren wir einfach alle aus:

In [12]:
geheim = 'KDOOR'
for k in range(1, 26):
    print(k, caesar(geheim, -k))

1 JCNNQ
2 IBMMP
3 HALLO
4 GZKKN
5 FYJJM
6 EXIIL
7 DWHHK
8 CVGGJ
9 BUFFI
10 ATEEH
11 ZSDDG
12 YRCCF
13 XQBBE
14 WPAAD
15 VOZZC
16 UNYYB
17 TMXXA
18 SLWWZ
19 RKVVY
20 QJUUX
21 PITTW
22 OHSSV
23 NGRRU
24 MFQQT
25 LEPPS


##### Beliebige Permutation

Wir müssen nicht um einen festen Betrag verschieben. Wir können jedem Buchstaben einen **beliebigen** anderen zuordnen, solange keine zwei Buchstaben auf denselben abgebildet werden. Eine solche Vertauschung des Alphabets heißt **Permutation**. Der Schlüssel ist das ganze Geheimalphabet.

In [13]:
import random

geheimalphabet = list(ALPHABET)
random.shuffle(geheimalphabet)
geheimalphabet = ''.join(geheimalphabet)
print(ALPHABET)
print(geheimalphabet)

def substitution(text, von, nach):
    ergebnis = ''
    for zeichen in text:
        if zeichen in von:
            ergebnis += nach[von.index(zeichen)]
        else:
            ergebnis += zeichen
    return ergebnis

geheim = substitution('GEHEIME NACHRICHT', ALPHABET, geheimalphabet)
print(geheim)
print(substitution(geheim, geheimalphabet, ALPHABET))    # zurück zum Klartext

ABCDEFGHIJKLMNOPQRSTUVWXYZ
ROTHWEAMJNGBSPFDUVLYIXCZQK
AWMWJSW PRTMVJTMY
GEHEIME NACHRICHT


Der Schlüsselraum besteht jetzt aus allen Permutationen der 26 Buchstaben. Für den ersten Buchstaben gibt es 26 Möglichkeiten, für den zweiten noch 25 usw. Das sind 26 · 25 · 24 · … · 1 = 26! (gesprochen: 26 Fakultät) Schlüssel.

In [14]:
import math
a = math.factorial(26)
print(f'{a}, {a:5.2e}')

403291461126605635584000000, 4.03e+26


In [16]:
anzahl = math.factorial(26)
print(f'{anzahl}, also etwa {anzahl:.2e}')

# Ein Computer probiert eine Milliarde Schlüssel pro Sekunde:
jahre = anzahl / 1e9 / 60 / 60 / 24 / 365
print(f'{jahre:.1e} Jahre')    # rund 13 Milliarden Jahre – so alt ist das Universum

403291461126605635584000000, also etwa 4.03e+26
1.3e+10 Jahre


Ein Brute-Force-Angriff ist also nicht mehr möglich. Trotzdem ist das Verfahren unsicher.

##### Häufigkeitsanalyse

In deutschen Texten kommen die Buchstaben unterschiedlich oft vor. Mit Abstand am häufigsten ist `E` (etwa 17 %), danach folgen `N`, `I`, `S`, `R`, `A` und `T`.

Bei einer monoalphabetischen Substitution wird jedes `E` durch denselben Buchstaben ersetzt. Dieser Buchstabe ist dann im Geheimtext genauso häufig wie das `E` im Klartext. Bei der **Häufigkeitsanalyse** zählen wir die Buchstaben im Geheimtext und vergleichen mit den normalen Häufigkeiten. Weitere Hilfen:

- Häufige kurze Wörter: `DER`, `DIE`, `UND`, `DAS`, `EIN`, `IST`
- Häufige Buchstabenpaare: `EN`, `ER`, `CH`, `EI`, `IE`, `SCH`

Mit etwas Ausprobieren findet man so die verwendete Permutation.

##### Beispiel

Der folgende Geheimtext wurde mit einer monoalphabetischen Substitution (beliebige Permutation) verschlüsselt. Umlaute wurden als `AE`, `OE`, `UE` geschrieben.

```
NUI NUKUMXBUQUIGZDK JXM VUIBI CIGIUFJI ZDQIGIQ XBBMXKQ VIGXIDNIGM
UDTAGPXMUADID LAIDDID JIZMI UDDIGJXBC WIDUKIG QILZDNID WIBMWIUM
ZICIGMGXKID WIGNID NXCIU QHUIBID FAPHZMIG DIMYWIGLI ZDN NXQ UDMIGDIM
IUDI WUFJMUKI GABBI KBIUFJYIUMUK KIWUDDM NIG QFJZMY HIGQAIDBUFJIG
NXMID UPPIG PIJG XD CINIZMZDK QUFJIGI HXQQWAIGMIG
VIGQFJBZIQQIBZDKQVIGTXJGID ZDN IUD CIWZQQMIG ZPKXDK PUM NUKUMXBID
PINUID JIBTID NXCIU NUI HGUVXMQHJXIGI YZ QFJZIMYID ZDN GUQULID UP
UDMIGDIM YZ VIGGUDKIGD
```

 

<img src='./haeufigkeit.png' width='500'>

Übung: Entziffere den Geheimtext mit dem [Solver-Tool](https://kryptografie.de/kryptografie/kryptoanalyse/solver-tool-zum-manuellen-knacken-von-monoalphabetischen-substitutions-chiffren.htm)

### Polyalphabetische Substitution: die Vigenère-Chiffre


Bei einer **polyalphabetischen Substitution** kann derselbe Klartextbuchstabe durch **verschiedene** Geheimtextbuchstaben ersetzt werden. Es kommt darauf an, an welcher Stelle er im Text steht.

Das bekannteste Beispiel ist die **Vigenère-Chiffre** aus dem 16. Jahrhundert. Sender und Empfänger vereinbaren ein **Schlüsselwort**. Es wird so oft wiederholt unter den Klartext geschrieben, bis es so lang ist wie der Klartext. Jeder Klartextbuchstabe wird dann mit der Cäsar-Chiffre verschlüsselt, und zwar um die Nummer des Schlüsselbuchstabens darunter (`A` = 0, `B` = 1, …, `Z` = 25).

Man kann das Ergebnis auch im **Vigenère-Quadrat** ablesen: Zeile = Schlüsselbuchstabe, Spalte = Klartextbuchstabe.

<img src='./vigenere.png' width='500'>




**Beachte:** Die beiden `N` im Klartext werden zu `G` und zu `Z`. Die beiden `I` werden zu `U` und zu `W`. Eine einfache Häufigkeitsanalyse über den ganzen Text funktioniert deshalb nicht mehr: Die Häufigkeiten werden „verwischt“.

[Vigenère-Chiffre](https://kryptografie.de/kryptografie/chiffre/vigenere.htm)



In [18]:
ALPHABET = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
def vigenere(text, schluessel, richtung=1):
    # richtung = 1: verschlüsseln, richtung = -1: entschlüsseln
    # text und schluessel bestehen nur aus Großbuchstaben A bis Z
    ergebnis = ''
    for i in range(len(text)):
        p = ALPHABET.index(text[i])
        k = ALPHABET.index(schluessel[i % len(schluessel)])    # Schlüsselwort wiederholen
        ergebnis += ALPHABET[(p + richtung * k) % 26]
    return ergebnis

geheim = vigenere('TREFFPUNKT', 'HUND')
print(geheim)                            # ALRIMJHQRN
print(vigenere(geheim, 'HUND', -1))      # TREFFPUNKT

ALRIMJHQRN
TREFFPUNKT


Die Vigenère-Chiffre galt rund 300 Jahre lang als unknackbar. Erst im 19. Jahrhundert fanden Charles Babbage und Friedrich Kasiski einen Angriff. 


### Angriff auf die Vigenère-Chiffre


Die Schwachstelle der Vigenère-Chiffre ist, dass sich das Schlüsselwort **wiederholt**. Hat das Schlüsselwort die Länge 5, dann wird der 1., 6., 11., 16., … Buchstabe mit derselben Verschiebung verschlüsselt. Der Angriff läuft in drei Schritten:

1. **Schlüssellänge bestimmen** (Kasiski-Test)
2. **Geheimtext in Teiltexte zerlegen:** Bei Schlüssellänge 5 kommen der 1., 6., 11., … Buchstabe in den ersten Teiltext, der 2., 7., 12., … in den zweiten usw.
3. **Jeden Teiltext einzeln knacken:** Jeder Teiltext ist eine Cäsar-Chiffre. Mit einer Häufigkeitsanalyse finden wir seine Verschiebung und damit einen Buchstaben des Schlüsselworts.

##### Schritt 1: Der Kasiski-Test

In deutschen Texten wiederholen sich oft Buchstabenfolgen, zum Beispiel `DER`, `EIN` oder `SCH`. Wenn zwei solche gleichen Folgen zufällig auf dieselbe Stelle im Schlüsselwort fallen, werden sie gleich verschlüsselt. Im Geheimtext entsteht dann zweimal dieselbe Folge.

Das passiert nur, wenn der **Abstand** der beiden Folgen ein **Vielfaches der Schlüssellänge** ist. Wir suchen also gleiche Folgen im Geheimtext, messen ihre Abstände und suchen einen gemeinsamen Teiler.

<img src='./kasiski.png' width='450'>



Übung: Entschlüssle den Geheimtext mit [Kasiski-Test](https://kryptografie.de/kryptografie/kryptoanalyse/kasiski-test.htm) und [Schlüsselrekonstruktion](https://kryptografie.de/kryptografie/kryptoanalyse/2-3-2-brechen-von-vigenere-mittels-schluesselrekonstruktion.htm)



    OQGRKJXVVEZOKLUPAEOTPNVPZEAKJAXQVKXCNTHZPEKLFLVPHVSZKJAEMPZHGMTZVSTWLLDMNUDLVPKT
    DAPBKOMTYBNPVPZPMOWYLMPNXCAKLEPAGUDLVPZVSWPPFLTVLGCWOOTERWSBFAEHXDITZXTVGIKTMHLO
    PZUJAWCGZLPTVPGOMOLKUMFLGMCEOLEIDLGFUFYXTAVLEWMPPFLTROTMMVCXCAEOHMMPOTEARHXEMTDN
    CLGUWTMXLKQIJYXYQOTXCJGZLPZCIXCIWJAOQGHGRZGPYPZYBKOMPPFXMTNXDKJPVVBGYWTMXLKDKJSN
    PAULEFVIUTNPXPZPVGYXRINAELVILTWAWUDYIERULZYLBWLGYLPTDLUFKJZMLJGKXDSNHKEMZAXDLWYV
    SDGYLNPKLWPVGINNPUATMMPKXDOGOXTUVLQEMULKDMVGMHQTKXCAVPFYMWUSPPPAXYRCOKSCPKXCBHHG
    OMPJALZNLLMIDITRMWUWQZKLWCQEODLAKZDTPGYTFAYPXXIPKBPTCLGRMFLLDKJSNPAULEHWTAXDJGZM
    TUOLGVIPUPPVPTTYLKLELMPNXVMPUMKMTSXRBOHGOMPNXSMKTMPFVPGPQPGXWVGAXTTVLQEMLLWPZVLB
    WBGEMHCTKXXQVKXCAGSUPVXLKDKJPXMCPNOPZUJAWCGZLPTVBGOSCUGOMUOTWJOPMPQPLKSIGBYTOMLB
    EACUTWGULZPJTVVSMPDXCLGUBYLGBMDKJLGEMZAXYQUAWPZDBVSAVHUPMOPMLJUATYLCTALMWMBRAVLG
    OIPHVSNQSZPVFPXMCEOLEIDLGYQUBGOZLLELMPNXCLGYZPPGPFEMZABDBFLLEWDLLDMTMNYSVPHYQGYM
    OQGZXCIPNKTNHOXFBGCXCEGUWPBOHGXWFLKYMXLKQIJYXYLKLTFKJTBEAEOGPTNLGNWOWNEMTUGTKJAB
    YDGYGFMPMMTOGYSPQVNXVVCJDEEGYWPVMVXYVGU


#### One Time Pad

Der Angriff auf die Vigenère-Chiffre funktioniert, weil sich das Schlüsselwort wiederholt und weil es meist ein sinnvolles Wort ist. Das **One-Time-Pad** (Einmalschlüssel) beseitigt genau diese Schwächen. Wir können es uns als Vigenère-Chiffre vorstellen, bei der für den Schlüssel drei Regeln gelten:

1. Der Schlüssel ist **genauso lang** wie die Nachricht. Er wiederholt sich also nie.
2. Der Schlüssel ist **echt zufällig**. Jeder Buchstabe wird unabhängig von den anderen zufällig gewählt.
3. Der Schlüssel wird **nur ein einziges Mal** benutzt (daher der Name *One-Time*) und danach vernichtet.

##### Warum ist das One-Time-Pad absolut sicher?

Eve kennt nur den Geheimtext. Sie könnte alle Schlüssel ausprobieren. Aber: Zu **jedem** Klartext der passenden Länge gibt es genau einen Schlüssel, der diesen Geheimtext erzeugt. Beim Ausprobieren kommen also **alle** sinnvollen Texte dieser Länge heraus, zum Beispiel `HALLO`, `HILFE`, `LIEBE` und `NEBEL`. Da der Schlüssel echt zufällig ist, ist jeder dieser Klartexte gleich wahrscheinlich. Der Geheimtext verrät **nichts** über den Klartext, egal wie viel Rechenleistung Eve hat.

##### Nachteile

Das One-Time-Pad ist in der Praxis sehr unhandlich:

- Für jede Nachricht braucht man einen neuen Schlüssel, der so lang ist wie die Nachricht.
- Der Schlüssel muss vorher auf einem **sicheren Weg** zum Empfänger gebracht werden, zum Beispiel bei einem persönlichen Treffen. Wenn es diesen sicheren Weg gibt, könnte man aber oft auch gleich die Nachricht selbst übergeben.
- Echte Zufallsbuchstaben kann man sich nicht merken. Sie müssen aufgeschrieben oder gespeichert und gut geschützt werden.

Eingesetzt wurde das One-Time-Pad deshalb nur dort, wo höchste Sicherheit wichtiger war als der Aufwand, zum Beispiel beim „heißen Draht“ zwischen Washington und Moskau im Kalten Krieg.



#### Kerckhoffs' Prinzip

Ein konkretes Verschlüsselungsverfahren besteht aus zwei Teilen:

- dem **Algorithmus**, also der Rechenvorschrift (z. B. „verschiebe jeden Buchstaben um *k* Stellen“),
- dem **Schlüssel** (z. B. *k* = 3).

Auguste Kerckhoffs formulierte 1883 einen Grundsatz, der bis heute gilt:

> Die Sicherheit eines Verschlüsselungsverfahrens darf **nur auf der Geheimhaltung des Schlüssels** beruhen, nicht auf der Geheimhaltung des Algorithmus.

Wir gehen also immer davon aus, dass der Angreifer das Verfahren kennt. Gründe dafür:

- **Ein Algorithmus lässt sich kaum geheim halten.** Er steckt in jedem Gerät und in jeder App. Man kann ihn durch Analyse der Software (*Reverse Engineering*), durch Spionage oder durch Verrat herausfinden.
- **Ein Schlüssel lässt sich leicht austauschen**, ein Algorithmus nicht. Wird ein Schlüssel bekannt, wählt man einfach einen neuen. Wird ein geheimer Algorithmus bekannt, ist das ganze Verfahren wertlos.
- **Öffentliche Verfahren können von vielen Fachleuten geprüft werden.** Schwachstellen werden so gefunden, bevor Angreifer sie ausnutzen. Moderne Verfahren wie AES wurden deshalb öffentlich ausgeschrieben und jahrelang untersucht.

Ein Vergleich: Wie ein Haustürschloss aufgebaut ist, ist allgemein bekannt. Die Tür ist trotzdem sicher, solange nur die Bewohner den passenden Schlüssel haben.

Wer sich dagegen auf ein geheimes Verfahren verlässt, betreibt *Security by Obscurity* (Sicherheit durch Verschleierung). Das gilt als schlechte Praxis.

#### Asymmetrische Verschlüsselung

##### Das Problem des Schlüsselaustauschs

Bei allen bisherigen Verfahren benutzen Sender und Empfänger **denselben** geheimen Schlüssel (**symmetrische Verschlüsselung**). Bevor Alice und Bob geheim kommunizieren können, müssen sie den Schlüssel also **sicher austauschen**. Aber wie, wenn sie sich nie persönlich treffen, etwa beim Online-Shopping?

Dazu kommt: Wenn jede Person aus einer Gruppe mit jeder anderen geheim kommunizieren will, braucht jedes Paar einen eigenen Schlüssel. Bei *n* Personen sind das *n* · (*n* − 1) / 2 Schlüssel.

##### Die Idee: ein Schlüsselpaar

Bei der **asymmetrischen Verschlüsselung** besitzt jede Person ein **Schlüsselpaar**:

- **Öffentlicher Schlüssel (Public Key):** darf jeder kennen. Er wird zum Beispiel auf einer Webseite veröffentlicht. Mit ihm kann man Nachrichten an diese Person **verschlüsseln**.
- **Privater Schlüssel (Private Key):** bleibt geheim und verlässt nie das eigene Gerät. Nur mit ihm kann man die Nachrichten wieder **entschlüsseln**.

Zum Verschlüsseln und Entschlüsseln werden also **verschiedene** Schlüssel benutzt. Daher der Name *asymmetrisch*.

**Vergleich mit einem Vorhängeschloss:** Bob verteilt an alle offene Vorhängeschlösser. Das ist sein öffentlicher Schlüssel. Jeder kann damit eine Kiste verschließen, denn zum Zuschnappen braucht man keinen Schlüssel. Öffnen kann die Kiste aber nur Bob, denn nur er hat den passenden Schlüssel. Das ist sein privater Schlüssel.

**Ablauf, wenn Alice Bob eine Nachricht schickt:**

1. Bob veröffentlicht seinen öffentlichen Schlüssel.
2. Alice verschlüsselt die Nachricht mit **Bobs öffentlichem** Schlüssel.
3. Bob entschlüsselt die Nachricht mit **seinem privaten** Schlüssel.

```
  Alice                                                         Bob
Klartext --verschlüsseln--> Geheimtext --entschlüsseln--> Klartext
     (Bobs öffentlicher Schlüssel)      (Bobs privater Schlüssel)
```

Eve kennt den öffentlichen Schlüssel auch. Damit kann sie aber nur verschlüsseln, nicht entschlüsseln. Den privaten Schlüssel kann man aus dem öffentlichen **praktisch nicht berechnen**, auch nicht mit sehr schnellen Computern. Ein vorheriger geheimer Schlüsselaustausch ist nicht mehr nötig.

##### Warum kann man den privaten Schlüssel nicht berechnen?

Asymmetrische Verfahren beruhen auf mathematischen Aufgaben, die in eine Richtung leicht und in die andere Richtung sehr schwer sind. Beim bekanntesten Verfahren, **RSA**, ist das die Zerlegung in Primfaktoren: Zwei große Primzahlen zu multiplizieren geht blitzschnell. Aus dem Produkt die beiden Primzahlen zurückzugewinnen, dauert dagegen sehr lange. Vereinfacht gesagt steckt das Produkt im öffentlichen Schlüssel und die beiden Primzahlen im privaten.

##### Hybride Verschlüsselung

Asymmetrische Verfahren sind rechenaufwendig und deutlich langsamer als symmetrische. In der Praxis kombiniert man deshalb beide (**hybride Verschlüsselung**):

1. Nur der kurze Sitzungsschlüssel wird **asymmetrisch** übertragen.
2. Die eigentliche Nachricht wird mit einem schnellen **symmetrischen** Verfahren und einem zufälligen Sitzungsschlüssel verschlüsselt.



### Verschlüsselung im Alltag

Verschlüsselung wird eingesetzt, um Daten vor unbefugtem Zugriff zu schützen. Ein wichtiger Anwendungsbereich ist die verschlüsselte Speicherung von Daten auf Computern, Smartphones oder USB-Sticks. Selbst wenn ein Gerät verloren geht, können die Daten ohne den richtigen Schlüssel nicht gelesen werden.

Auch bei der Übertragung von Daten im Internet spielt Verschlüsselung eine große Rolle. Webseiten mit HTTPS verschlüsseln die Verbindung zwischen Browser und Server, sodass Passwörter oder andere persönliche Informationen nicht einfach mitgelesen werden können. Ebenso verwenden viele Messenger-Dienste eine Ende-zu-Ende-Verschlüsselung, damit nur die beteiligten Personen die Nachrichten lesen können. Moderne WLANs sind mit WPA2 oder WPA3 verschlüsselt. Andere Personen in Reichweite des Funknetzes können den Datenverkehr dann nicht einfach mitlesen.



### Personenbezogene Daten

**Personenbezogene Daten** sind alle Informationen, die sich einer bestimmten Person zuordnen lassen: Name, Adresse und Geburtsdatum, aber auch Standort, IP-Adresse, Interessen oder das Surfverhalten. Für viele Unternehmen sind diese Daten sehr wertvoll, zum Beispiel für personalisierte Werbung. Sie werden automatisch erhoben, zusammengeführt und ausgewertet.

#### Cookies

**Cookies** sind kleine Textdateien, die eine Webseite im Browser speichert und bei einem erneuten Besuch wieder ausliest. Damit lassen sich unter anderem Login-Zustände, Spracheinstellungen, der Warenkorb, besuchte Seiten und eine eindeutige **Nutzer-ID** speichern.

- **Notwendige Cookies** sind für die Grundfunktion der Seite erforderlich, z. B. damit man eingeloggt bleibt oder der Warenkorb nicht verschwindet.
- **Tracking-Cookies** verfolgen das Verhalten über **mehrere Webseiten** hinweg. Sie stammen meist von **Drittanbietern**, zum Beispiel Werbenetzwerken.

**Wie Tracking funktioniert:** Ein Werbenetzwerk ist auf sehr vielen Webseiten eingebunden, etwa durch eine Werbeanzeige oder einen „Gefällt mir“-Knopf. Beim ersten Besuch speichert es im Browser ein Cookie mit einer ID. Auf jeder weiteren Seite, auf der es eingebunden ist, liest es diese ID wieder aus und erfährt so, welche Seiten derselbe Browser besucht. Aus allen Besuchen entsteht ein **Nutzungsprofil**.

Das Werbenetzwerk kennt vielleicht nicht den Namen. Aber sobald man sich auf **einer** der Seiten mit Namen oder E-Mail-Adresse anmeldet, lässt sich das ganze Profil einer Person zuordnen.

Neben Cookies gibt es weitere Tracking-Techniken, z. B. **Fingerprinting**: Aus vielen Merkmalen des Geräts (Browser, Bildschirmgröße, installierte Schriftarten, …) wird ein „Fingerabdruck“ berechnet, der das Gerät auch ohne Cookie wiedererkennt.

#### Geodaten


Smartphones bestimmen ihren Standort über **GPS** (Satellitensignale), ersatzweise über nahe **WLAN-Netze** oder die Entfernung zu **Mobilfunkmasten**. Viele Apps fragen nach dem Standort, und manche speichern ihn regelmäßig. Über die Zeit gesammelte Standortdaten ergeben ein **Bewegungsprofil**. Daraus lassen sich Wohnort, Schule oder Arbeitsplatz, Tagesablauf und regelmäßig besuchte Orte ableiten.


#### Möglichkeiten zum Schutz personenbezogener Daten

Es gibt verschiedene Möglichkeiten, das Sammeln personenbezogener Daten einzuschränken. Viele Browser bieten einen anonymen Modus oder ermöglichen das Blockieren von Tracking-Cookies. Dadurch können Webseiten Nutzer schlechter verfolgen.

Außerdem sollte man die Berechtigungen von Apps regelmäßig überprüfen. Viele Apps verlangen Zugriff auf Kamera, Mikrofon oder Kontakte, obwohl dies nicht immer notwendig ist. Solche Rechte sollten nur erteilt werden, wenn sie wirklich gebraucht werden.

Auch die Standortfreigabe sollte bewusst genutzt werden. Wenn der Standort nicht benötigt wird, kann die Freigabe deaktiviert werden. Dadurch werden weniger Bewegungsdaten gesammelt und gespeichert.